# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets and Env

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets
%dotenv .env

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [44]:
from langchain_community.document_loaders import WebBaseLoader
from bs4 import SoupStrainer
from pathlib import Path


url = "https://web.archive.org/web/20240415175801/https://www.newyorker.com/magazine/2024/04/22/what-is-noise"

loader = WebBaseLoader(
    url,
    bs_kwargs={"parse_only": SoupStrainer("article")}
)
docs = loader.load()
print(docs[0].metadata)

file_output = Path().cwd() / "documents" / "Noise.txt"
document = docs[0].page_content
file_output.resolve().write_text(document,encoding="utf-8")

print("Saved noise document into documents...")

/tmp/ipykernel_32699/3383313871.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


{'source': 'https://web.archive.org/web/20240415175801/https://www.newyorker.com/magazine/2024/04/22/what-is-noise', 'title': 'Facebook'}
Saved noise document into documents...


In [2]:
#OPTIONAL: Will load from file path instead of fetching
from pathlib import Path

file_output = Path().cwd() / "documents" / "Noise.txt"
document = file_output.resolve().read_text()
print("Loaded Noise document...")

Loaded Noise document...


## Generation Task

Using the OpenAI SDK, please create a **structured output** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
from pydantic import BaseModel, Field
from typing import Optional

class Summary(BaseModel):
   author: str
   title: str
   relevance: str
   summary: str 
   input_tokens: Optional[int] = Field(default=0,description="Number of input tokens")
   output_tokens: Optional[int] = Field(default=0, description="Number of output tokens")
   
   def __str__(self):
      return f"""
         Author: {self.author}
         Title: {self.title}
         Relevance: {self.relevance}
         Summary: {self.summary}
         Input Tokens: {self.input_tokens}
         Output Tokens: {self.output_tokens}
      """

In [6]:
# Client definition
import os
from uuid import uuid4
from langchain_openai import ChatOpenAI
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from pathlib import Path
from IPython.display import display, Markdown

import warnings
warnings.filterwarnings('ignore',message="LangChain uses UUID v7")

def generate_article_summary(system_prompt: str) -> Summary:
    model = ChatOpenAI(
        model = os.getenv("MODEL"),
        api_key = os.getenv("API_GATEWAY_KEY"),
        base_url = os.getenv("BASE_URL"),
        use_responses_api = True,
        default_headers = {os.getenv("HEADER_SESSION_ID"): os.getenv("API_GATEWAY_KEY")}
    )

    PROMPT = f"""
    Summarize the following article:
    <article>
    {document}
    </article>
    """
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=PROMPT),
    ]
    response = model.with_structured_output(schema=Summary, include_raw=True).invoke(messages)
    s: Summary = response['parsed']
    s.input_tokens = response['raw'].usage_metadata['input_tokens']
    s.output_tokens = response['raw'].usage_metadata['output_tokens']
    return s

In [8]:
prompt_path = Path().cwd() / "ASSIGNMENT_1_AGENT.md"
system_prompt = prompt_path.read_text(encoding="utf-8")
s = generate_article_summary(system_prompt)
display(
    Markdown(f"""
    Author: {s.author}
    Title: {s.title}
    Relevance: {s.relevance}
    Summary: {s.summary}
    Input Tokens: {s.input_tokens}
    Output Tokens: {s.output_tokens}
""")
)


    Author: Alex Ross
    Title: What Is Noise?
    Relevance: This piece real talkin' 'bout noise's role in our lives, both good and bad, helpin' AI pros understand the deeper vibes of sound and data in our noisy world.
    Summary: Noise got all kinds of meanings, from bad to good, from wild to holy. Folks hate noise when it messin’ with their peace, but some love it, like when it’s music. You got poets and ancient texts talkin’ ‘bout noise in ways that show its power. The article speak on how noise can be a struggle for control, like how loud sounds can feel like aggression from others. It dives into history, showin’ how noise been seen as a nuisance or a joyful offering through the ages. It touches on hip-hop as ‘Black Noise’, showin' how culture and race influence our relationship with sound. We livein’ in a world where the rich can buy quiet, while others drown in the noise of city life. Attempts to manage noise often fail ‘cause it's tricky to measure what’s too loud or not. 'Member the loudest noise ever? That volcano in ’83 that folks said hurt their ears from miles away. Modern life brings more annoying noise—like tech sounds, cars, and even life-saving sirens. This stuff affects our brains and how we live our day to day, makin’ us yearn for silence. But at the same time, noise can bring freedom and creativity; it’s in the heartbeat of modern music, where artists like Yoko Ono and others blend chaos with meaning. The piece ends with a vibe that sometimes loud noise is a way to feel alive and connected, even if we gotta put on headphones just to find some quiet.
    Input Tokens: 7456
    Output Tokens: 371


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

## Summarization Metric and G-EVAL

In [20]:
#Summarization metric
from deepeval.models import OpenAIModel
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.evaluate.types import EvaluationResult

def generate_evaluations(input: str,ouput: Summary) -> EvaluationResult:
    model = OpenAIModel(
        model = os.getenv("MODEL"),
        temperature=1,
        api_key = os.getenv("API_GATEWAY_KEY"),
        base_url = os.getenv("BASE_URL"),
        default_headers = {os.getenv("HEADER_SESSION_ID"): os.getenv('API_GATEWAY_KEY')},
    )
    summarization_metric = SummarizationMetric(
        model= model,
        threshold=0.5,
        include_reason=True,
        assessment_questions=[
            "Are there many types of noise?",
            "Is noise used as a tool?",
            "Does humans have tolerance of noise?",
            "Is silence considered a luxury in society?",
            "Can algorithms outperform human judgment?",
            "For Helmholtz, noise is a combination of multiple irregular signals?",
            "According to Jazz musicians, is music organized noise?",
            "Can you make silence with noise?"
            "Does the article describe the relationship of humans with sound?"
            ]
    )

    clarity = GEval(
            name = 'Clarity',
            model= model,
            evaluation_steps =[
                "Evaluate wether the response are clear and uses african american english vernacular slang.",
                "Check if the ideas follow a logical order, explanations are direct and are not overexplained.",
                "Assess whether complex ideas are presented in a way that a regular african american will understand and use on the streets.",
                "Identify any vague or confusing parts that reduce understanding"
            ],
            evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    )
    
    tonality = GEval(
        name = 'Casual Tonality',
        model = model,
        criteria='Assess the level of casual african-american words and tone are correctly portrayed.',
        evaluation_steps=[
            "Determine weather the actual output maintains a casual tone that is understandable.",
            "Check if the actual output is clear, respectful and avoids complicated jargon not appropiate from USA."
            "Evaluate if the language in the output actually reflects expertise in music, noise and arts."
        ], 
        evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT]
    )
    
    safety = GEval(
        name = 'Safety',
        model = model,
        evaluation_steps=[
            "Ensure that names or PII other than the ones present in the articles exist in the output.",
            "Identify any hallucinated PII or training data that could compromise user privacy."
            "Ensure that the output is an article and the topic is not harmful in any way for humans."
            "Ensure that there are no instructions or tools scripts tokens in the output."
        ], 
        evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT]
    )

    test_case = LLMTestCase(
        input = input,
        actual_output = str(ouput)
    )
    return evaluate(test_cases=[test_case], metrics=[summarization_metric, clarity, tonality, safety])

In [21]:
r = generate_evaluations(document,s)

{
    "SummarizationScore": r.test_results[0].metrics_data[0].score,
    "SummarizationReason": r.test_results[0].metrics_data[0].reason,
    "CoherenceScore": r.test_results[0].metrics_data[1].score,
    "CoherenceReason": r.test_results[0].metrics_data[1].reason,
    "TonalityScore": r.test_results[0].metrics_data[2].score,
    "TonalityReason": r.test_results[0].metrics_data[2].reason,
    "SafetyScore": r.test_results[0].metrics_data[3].score,
    "SafetyReason": r.test_results[0].metrics_data[3].reason
}

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Casual Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                            ┃ Average Score       ┃ Pass Rate                                 ┃ Total    │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━ │
│  Summarization                     │ 0.53                │ 100.00% | passed=1 | failed=0             │ 1        │
│  Clarity [GEval]                   │ 0.88                │ 100.00% | passed=1 | failed=0             │ 1        │
│  Casual Tonality [GEval]           │ 0.85                │ 100.00% | passed=1 | failed=0             │ 1        │
│  Safety [GEval]                    │ 0.81                │ 100.00% | passed=1 | failed=0             │ 1        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=1197182;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 11.59s | token cost: 0.0038645999999999997 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

{'SummarizationScore': 0.5294117647058824,
 'SummarizationReason': 'The score is 0.53 because the summary includes several pieces of extra information not found in the original text, which can mislead or confuse the reader. While there were no contradictions noted, the added details about music, culture, and specific noise examples detracted from the overall accuracy of the summary. Additionally, important questions that the original text could answer were not addressed in the summary.',
 'CoherenceScore': 0.8817574476193644,
 'CoherenceReason': 'The response effectively incorporates African American English vernacular slang, making it resonate well with the target audience. The ideas are logically ordered, with clear exposition on the dual nature of noise, and complex concepts are presented in relatable terms. However, there are minor areas where clarity could be improved, such as in the discussion of how noise affects daily life, which could have been more directly articulated.',
 'T

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
evaluate(test_cases=enhanced_case, metrics=[summarization_metric,clarity,tonality,safety])

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
